In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import xarray as xr

import eradiate
from eradiate import fresolver

eradiate.set_mode("mono")
sns.set_theme(style="ticks")

In [ ]:
ds1 = fresolver.load_dataset("aerosol/govaerts_2021-desert.nc")
ds2 = fresolver.load_dataset(
    "aerosol/aeronet_sahara_spherical_RAMIA_GENERIC_extrapolated.nc"
)

ds1["sigma_t"] *= 1e-3
ds1["w"] = ds1["w"] * 1e-3
ds1["sigma_t"].plot(marker=".", label="govaerts...")
ds2["sigma_t"].plot(marker=".", ls=":", label="rami4atm")
plt.legend()

In [ ]:
ds1 = fresolver.load_dataset("aerosol/govaerts_2021-continental.nc")
ds2 = fresolver.load_dataset("aerosol/govaerts_2021-continental-extrapolated.nc")
ds3 = fresolver.load_dataset(
    "aerosol/aeronet_continental_clean_spherical_RAMIA_GENERIC.nc"
)
ds4 = fresolver.load_dataset(
    "aerosol/aeronet_continental_clean_spherical_RAMIA_GENERIC_extrapolated.nc"
)

ds1["sigma_t"] *= 1e-3
ds1["w"] = ds1["w"] * 1e-3
ds1["sigma_t"].plot(marker=".", label="govaerts")
ds2["sigma_t"].plot(marker=".", ls="--", label="govaerts")
ds3["sigma_t"].plot(marker=".", ls=":", label="rami4atm")
ds4["sigma_t"].plot(marker=".", ls="", label="rami4atm")
plt.legend()

In [ ]:
# Conversion of RAMI4ATM datasets
DATA = {
    "govaerts_2021-desert": {
        "source": "govaerts_2021-desert",
        "attrs": {
            "title": "RAMI4ATM Desert Aerosol (legacy, unpolarized)",
            "references": "https://github.com/scottprahl/miepython/",
            "comment": "Individual Debye-Lorenz-Mie scattering computations with miepython v2.3.0, "
            "radius-averaged and concatenated along the spectral axis.",
        },
    },
    "govaerts_2021-desert-extrapolated": {
        "source": "govaerts_2021-desert-extrapolated",
        "attrs": {
            "title": "RAMI4ATM Desert Aerosol (legacy, unpolarized, extrapolated)",
            "references": "https://github.com/scottprahl/miepython/",
            "comment": "Individual Debye-Lorenz-Mie scattering computations with miepython v2.3.0, "
            "radius-averaged and concatenated along the spectral axis.\n"
            "Original dataset extrapolated to cover the [280, 350] nm.",
        },
    },
    "govaerts_2021-continental": {
        "source": "govaerts_2021-continental",
        "attrs": {
            "title": "RAMI4ATM Continental Aerosol (legacy, unpolarized)",
            "references": "https://github.com/scottprahl/miepython/",
            "comment": "Individual Debye-Lorenz-Mie scattering computations with miepython v2.3.0, "
            "radius-averaged and concatenated along the spectral axis.",
        },
    },
    "govaerts_2021-continental-extrapolated": {
        "source": "govaerts_2021-continental-extrapolated",
        "attrs": {
            "title": "RAMI4ATM Continental Aerosol (legacy, unpolarized, extrapolated)",
            "references": "https://github.com/scottprahl/miepython/",
            "comment": "Individual Debye-Lorenz-Mie scattering computations with miepython v2.3.0, "
            "radius-averaged and concatenated along the spectral axis.\n"
            "Original dataset extrapolated to cover the [280, 350] nm.",
        },
    },
    "aeronet_continental_clean_spherical_RAMIA_GENERIC_extrapolated": {
        "source": "aeronet_continental_clean_spherical_RAMIA_GENERIC_extrapolated",
        "attrs": {
            "title": "RAMI4ATM Continental Aerosol (extrapolated)",
            "references": "https://github.com/scottprahl/miepython/",
            "comment": "Individual Debye-Lorenz-Mie scattering computations with miepython v2.3.0, "
            "radius-averaged and concatenated along the spectral axis.\n"
            "Original dataset extrapolated to cover the [280, 350] nm.",
        },
    },
    "aeronet_sahara_spherical_RAMIA_GENERIC_extrapolated": {
        "source": "aeronet_sahara_spherical_RAMIA_GENERIC_extrapolated",
        "attrs": {
            "title": "RAMI4ATM Desert Aerosol (extrapolated)",
            "references": "https://github.com/scottprahl/miepython/",
            "comment": "Individual Debye-Lorenz-Mie scattering computations with miepython v2.3.0, "
            "radius-averaged and concatenated along the spectral axis.\n"
            "Original dataset extrapolated to cover the [280, 350] nm.",
        },
    },
}


def convert(src, **kw_attrs):
    ds = fresolver.load_dataset(f"aerosol/{src}.nc")
    # Fix for old ill-formatted files
    if ds["phase"].attrs["units"] == "str^-1":
        ds["phase"].attrs["units"] = "1 / sr"

    for k in ["convention", "Conventions", "comment", "source"]:
        try:
            del ds.attrs[k]
        except KeyError:
            continue
    ds.attrs["history"] = ds.attrs["history"].split(" - ")[0] + " - Dataset creation"
    ds.attrs.update(kw_attrs)

    result = eradiate.data.convert.aer_v1_to_aer_core_v2(ds)
    # display(result)

    return result


for dst in DATA.keys():
    src = DATA[dst]["source"]
    attrs = DATA[dst]["attrs"]
    ds = convert(src, **attrs)
    ds.to_netcdf(f"{dst}-aer_core_v2.nc")

In [ ]:
xr.load_dataset("soot.mie.cdf")

In [ ]:
DATA = {
    "soot.mie": {
        "sel": {"nhum": 0},
        "source": "soot.mie",
        "attrs": {},
    },
}


def convert(src, sel: dict, **kw_attrs):
    ds = fresolver.load_dataset(f"{src}.cdf")
    ds = ds.sel(sel)

    result = eradiate.data.convert.libradtran_to_aer_core_v2(ds)
    display(result)

    return result


for dst in DATA.keys():
    src = DATA[dst]["source"]
    sel = DATA[dst].get("sel", {})
    attrs = DATA[dst]["attrs"]
    ds = convert(src, sel, **attrs)
    ds.to_netcdf(f"{dst}-aer_core_v2.nc")

In [ ]:
import numpy as np

np.allclose(ds4["phase"].isel(phamat=0), ds4["phase"].isel(phamat=2))

In [ ]:
ds4["phase"].isel(phamat=0, drop=True).isel(w=8).plot(
    x="theta", ls=":", marker=".", label="11"
)
ds4["phase"].isel(phamat=1, drop=True).isel(w=8).plot(
    x="theta", ls=":", marker=".", label="12"
)
ds4["phase"].isel(phamat=2, drop=True).isel(w=8).plot(
    x="theta", ls=":", marker=".", label="33"
)
ds4["phase"].isel(phamat=3, drop=True).isel(w=8).plot(
    x="theta", ls=":", marker=".", label="34"
)
plt.yscale("symlog", linthresh=1e-3)
bound = np.max(np.abs(plt.ylim()))
plt.ylim(-bound, bound)
plt.legend(ncols=4)

In [ ]:
import xarray as xr

import eradiate

ds = xr.open_dataset("REF_UB.nc")
ds = eradiate.data.convert.aer_v1_to_aer_core_v2(ds)
ds.to_netcdf("REF_UB-aer_core_v2.nc")